In [0]:
from collections import defaultdict
from pathlib import Path
from typing import Dict, List
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as f
from torch.utils.data import Dataset, DataLoader, IterableDataset
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, ArrayType, StringType, DoubleType
from pyspark.ml.feature import StringIndexer
import pyarrow.parquet as pq
from itertools import chain
import pickle
import glob
import os
import shutil
import json
from time import perf_counter

In [0]:
def resolve_local_result_dir(path_str):
    """Convert dbfs:/ paths to local driver file-system paths."""
    if path_str.startswith("dbfs:/"):
        return "/dbfs/" + path_str[len("dbfs:/"):].lstrip("/")
    return path_str


def apply_shard_filter(df, shard_id, num_shards):
    """Filter a dataframe to the current shard based on shard_id or stable hash fallback."""
    if int(num_shards) <= 1:
        return df

    if "shard_id" in df.columns:
        return df.where(F.col("shard_id") == F.lit(int(shard_id)))

    # Fallback for legacy tables without shard_id column
    return df.where(F.pmod(F.xxhash64("user_id"), F.lit(int(num_shards))) == F.lit(int(shard_id)))


def apply_partition_date_filter_if_exists(df, run_date):
    """Apply partition_date filter only for tables that contain that column."""
    if "partition_date" in df.columns:
        return df.where(F.col("partition_date") == run_date)
    return df


def normalize_shard_id(raw_shard_id, num_shards, shard_id_base):
    """Normalize pipeline shard ids to the zero-based ids used by the Spark shard column."""
    if int(num_shards) < 1:
        raise ValueError(f"num_shards must be >= 1, got {num_shards}")

    shard_id_base = str(shard_id_base or "0").strip().lower()
    if shard_id_base not in {"0", "1"}:
        raise ValueError(
            f"shard_id_base must be '0' or '1', got {shard_id_base}"
        )

    if shard_id_base == "0":
        if raw_shard_id < 0 or raw_shard_id >= num_shards:
            raise ValueError(
                f"shard_id must be in [0, {num_shards - 1}] when shard_id_base=0, got {raw_shard_id}"
            )
        return raw_shard_id

    if raw_shard_id < 1 or raw_shard_id > num_shards:
        raise ValueError(
            f"shard_id must be in [1, {num_shards}] when shard_id_base=1, got {raw_shard_id}"
        )
    return raw_shard_id - 1


# -----------------------------
# Dataset construction
# -----------------------------
def load_user_product_spark_dataframes(spark, db, table_names, run_date):
    """Load inference dataframes for a specific date."""
    user_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['mmc_user_processed_features']}
        WHERE partition_date = '{run_date}'
    """)
    
    product_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['mmc_product_processed_features']}
        WHERE partition_date = '{run_date}'
    """)
    
    # # Cache since we'll use them later
    # user_df.cache()
    # product_df.cache()
   
    return user_df, product_df

def load_candidate_spark_dataframes(spark, db, table_names, run_date):
    """Load inference dataframes for a specific date."""
    candidate_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['mmc_candidates']}
        WHERE partition_date = '{run_date}'
    """)
   
    return candidate_df

def build_shard_candidate_dataframe(df, run_date, shard_id=0, num_shards=1):
    df = apply_partition_date_filter_if_exists(df, run_date)
    df = apply_shard_filter(df, shard_id, num_shards)

    # Build shard-local row index once so chunk boundaries are correct inside each shard.
    df = df.withColumn(
        "__local_row_id",
        F.row_number().over(Window.orderBy(F.monotonically_increasing_id()))
    )
    return df

def load_candidate_chunk(candidate_df, chunk_start_idx, chunk_size):
    start_row = int(chunk_start_idx) + 1
    end_row = int(chunk_start_idx + chunk_size)
    return candidate_df.where(
        (F.col("__local_row_id") >= F.lit(start_row))
        & (F.col("__local_row_id") <= F.lit(end_row))
    ).drop("__local_row_id")

def process_inference_dataframe(df, id_col, cat_cols, num_cols):
    """
    Process inference data using fitted pipeline.
    
    Args:
        df: Input dataframe
        id_col: ID column name
        cat_cols: List of categorical column names
        num_cols: List of numerical column names
    
    Returns:
        Processed dataframe and ID decoder mapping
    """
    df_proc = df
    # Create temporary integer encoding for IDs (for tensor compatibility)
    # This is different from the model input encoding - it's just for indexing
    
    indexer = StringIndexer(
        inputCol=id_col,
        outputCol=id_col + "_temp_idx",
        handleInvalid="keep"   # avoid failures if new IDs appear
    )

    # Fit the indexer (distributed operation)
    model = indexer.fit(df)
    # Transform the dataframe
    df_proc = model.transform(df)
    
    # Extract decoder mapping: index → original ID
    # labelsArray is an ordered list: index -> string
    labels = model.labels  # already collected efficiently
    decode_id_map = {i: lbl for i, lbl in enumerate(labels)}

    # Select final columns
    select_cols = [
        id_col,
        id_col + '_temp_idx',
        id_col + "_idx",
        *[f"{c}_idx" for c in cat_cols],
        *[f"{c}_norm" for c in num_cols]
    ]
    
    df_proc = df_proc.select(*select_cols)
    
    return df_proc, decode_id_map

def process_user_product_feature_data(user_df, product_df,
                            user_id_col, product_id_col,
                            user_cat_cols, user_num_cols,
                            product_cat_cols, product_num_cols):
    # Process user relavant columns
    user_df_proc, decode_uid_map = process_inference_dataframe(
        user_df,
        user_id_col,
        user_cat_cols,
        user_num_cols
    )
    print("User table process completed")

    # Process product relavant columns
    product_df_proc, decode_pid_map = process_inference_dataframe(
        product_df,
        product_id_col,
        product_cat_cols,
        product_num_cols
    )
    print("product table process completed")

    # Get statistics (useful for validation)
    n_users = user_df_proc.select(F.countDistinct(user_id_col + "_temp_idx")).first()[0]
    n_products = product_df_proc.select(F.countDistinct(product_id_col + "_temp_idx")).first()[0]

    return {
        'n_users': n_users,
        'n_products': n_products,
        'user_df': user_df_proc,
        'product_df': product_df_proc,
        'uid_decoder': decode_uid_map,
        'pid_decoder': decode_pid_map
    }

def join_ranking_inference_data(user_df, product_df, candidate_df, user_id_col, product_id_col):
    """
    Join candidate set data and feature data for inference
    """
    # Repartition
    candidate_df = candidate_df.repartition(user_id_col, product_id_col)
    user_df = user_df.repartition(user_id_col)
    product_df = product_df.repartition(product_id_col)
    
    # Optimized join: broadcast smaller table if possible
    # PySpark will auto-broadcast if one side is small enough
    joined_df = (candidate_df
                .join(user_df, user_id_col, "inner")
                .join(product_df, product_id_col, "inner"))

    return joined_df

In [0]:
def load_and_merge_results(result_dir):
    result_dir = resolve_local_result_dir(result_dir)
    res_files = sorted(glob.glob(f"{result_dir}/result_chunk_*.pkl"))
    results = [pickle.load(open(f, 'rb')) for f in res_files]
    merged_res = {key: [d[key] for d in results] for key in results[0]}

    return merged_res

def load_decoders(result_dir):
    result_dir = resolve_local_result_dir(result_dir)
    uid_decoder = pickle.load(open(Path(result_dir) / "uid_decoder.pkl", 'rb'))
    pid_decoder = pickle.load(open(Path(result_dir) / "pid_decoder.pkl", 'rb'))
    return uid_decoder, pid_decoder

def clear_result_dir(result_dir):
    result_dir = resolve_local_result_dir(result_dir)
    result_path = Path(result_dir)
    if not result_path.exists():
        print(f"Result directory does not exist yet: {result_dir}")
        return

    removed_entries = 0
    for child in result_path.iterdir():
        try:
            if child.is_dir():
                shutil.rmtree(child)
            else:
                child.unlink(missing_ok=True)
            removed_entries += 1
        except OSError as e:
            print(f"Warning: could not delete {child}: {e}")

    print(f"Cleared {removed_entries} existing result artifacts from {result_dir}")

In [0]:
class PandasRankingDataset(IterableDataset):
    def __init__(self, data_df, 
                 user_id_int_col, product_id_int_col,
                 user_id_col, product_id_col,
                 user_cat_cols, user_num_cols,
                 product_cat_cols, product_num_cols,
                 batch_size=1024):
        super().__init__()
        self.data_df = data_df
        self.user_id_col, self.product_id_col = user_id_col, product_id_col
        self.user_id_int_col, self.product_id_int_col = user_id_int_col, product_id_int_col
        self.user_cat_cols, self.user_num_cols = user_cat_cols, user_num_cols
        self.product_cat_cols, self.product_num_cols = product_cat_cols, product_num_cols
        self.batch_size = batch_size

    def __iter__(self):
        # Iterate over the Pandas DataFrame in chunks
        num_rows = len(self.data_df)
        
        for i in range(0, num_rows, self.batch_size):
            # Slice the Pandas DataFrame to get a batch
            batch = self.data_df.iloc[i : i + self.batch_size]
            
            # Convert columns to tensors
            # Note: .values is faster than .to_numpy() for simple columns
            user_id = torch.tensor(batch[self.user_id_col].values, dtype=torch.long)
            user_id_int = torch.tensor(batch[self.user_id_int_col].values, dtype=torch.long)

            product_id_int = torch.tensor(batch[self.product_id_int_col].values, dtype=torch.long)
            product_id = torch.tensor(batch[self.product_id_col].values, dtype=torch.long)

            # Categorical features
            user_cat = torch.stack([
                torch.tensor(batch[col].values, dtype=torch.long)
                for col in self.user_cat_cols
            ], dim=1) if self.user_cat_cols else torch.empty((user_id.size(0), 0), dtype=torch.long)

            # Numerical features
            user_num = torch.stack([
                torch.tensor(batch[col].values, dtype=torch.float32)
                for col in self.user_num_cols
            ], dim=1) if self.user_num_cols else torch.empty((user_id.size(0), 0), dtype=torch.float32)

            product_cat = torch.stack([
                torch.tensor(batch[col].values, dtype=torch.long)
                for col in self.product_cat_cols
            ], dim=1) if self.product_cat_cols else torch.empty((product_id.size(0), 0), dtype=torch.long)
            
            product_num = torch.stack([
                torch.tensor(batch[col].values, dtype=torch.float32)
                for col in self.product_num_cols
            ], dim=1) if self.product_num_cols else torch.empty((product_id.size(0), 0), dtype=torch.float32)      

            yield {
                "user_id": user_id,
                "user_id_int": user_id_int,
                "user_cat": user_cat,
                "user_num": user_num,
                "product_id": product_id,
                "product_id_int": product_id_int,
                "product_cat": product_cat,
                "product_num": product_num
            }

In [0]:
def create_data_loader(data_source, user_id_col, product_id_col, 
                       user_cat_cols, user_num_cols, 
                       product_cat_cols, product_num_cols, 
                       batch_size):
    """
    Args:
        data_source: A Pandas DataFrame containing the chunk of data.
    """
    dataset = PandasRankingDataset(
        data_df=data_source,
        user_id_int_col=user_id_col+'_temp_idx',
        product_id_int_col=product_id_col+'_temp_idx',
        user_id_col=user_id_col+'_idx', 
        product_id_col=product_id_col+'_idx',
        user_cat_cols=[c+'_idx' for c in user_cat_cols],
        user_num_cols=[c+'_norm' for c in user_num_cols],
        product_cat_cols=[c+'_idx' for c in product_cat_cols],
        product_num_cols=[c+'_norm' for c in product_num_cols],
        batch_size=batch_size
    )

    # Num_workers must be 0 for simple Pandas iteration in main process
    data_loader = DataLoader(dataset, batch_size=None, num_workers=0)
    return data_loader

In [0]:
# -----------------------------
# Model: shared trunk + heads
# -----------------------------
class MLP(nn.Module):
    """
    Multi-Layer Perceptron (MLP) used to process concatenated feature vectors.
    Consists of several linear layers with ReLU activations and dropout for regularization.
    """
    def __init__(self, in_dim, hidden_dims=(256,128), dropout=0.2):
        super().__init__()
        layers = []  # List to hold layers
        prev = in_dim  # Track input dimension for each layer
        for h in hidden_dims:
            # Add a Linear layer, followed by ReLU and Dropout
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h  # Update input dimension for next layer
        self.net = nn.Sequential(*layers)  # Combine layers into a sequential model

    def forward(self, x):
        # Forward pass through the MLP
        return self.net(x)

def _last_linear_out_features(seq):
    """
    Utility to get output dimension of last Linear layer in a Sequential.
    """
    for l in reversed(seq):
        if isinstance(l, nn.Linear): return l.out_features
    raise ValueError("No Linear found")

def calibrate_probability(p_pred, alpha: float):
    """
    Calibrate probability after training based on negative downsampling.
    Args:
        p_pred: raw sigmoid probability (tensor)
        alpha: global downsampling ratio (0 < alpha < 1)
    Returns:
        Calibrated probability tensor
    """
    return (alpha * p_pred) / ((1 - p_pred) + alpha * p_pred)

class MultiTaskRanker(nn.Module):
    def __init__(self, uid_vocab, uid_dim, user_cat_dims, user_num_dim,
                 pid_vocab, pid_dim, product_cat_dims, product_num_dim,
                 hidden_dims=(256,128), dropout=0.2, alpha=1):
        super().__init__()
        # Embedding for user ID (with padding index 0)
        self.user_id_emb = nn.Embedding(uid_vocab+1, uid_dim, padding_idx=0)
        # Embeddings for each categorical user feature
        self.user_cat_embs = nn.ModuleList([nn.Embedding(v+1, d, padding_idx=0) for v, d in user_cat_dims])

        # Embedding for product ID (with padding index 0)
        self.product_id_emb = nn.Embedding(pid_vocab+1, pid_dim, padding_idx=0)
        # Embeddings for each categorical product feature
        self.product_cat_embs = nn.ModuleList([nn.Embedding(v+1, d, padding_idx=0) for v, d in product_cat_dims])

        u_dim = uid_dim + sum(d for _, d in user_cat_dims) + user_num_dim
        p_dim = pid_dim + sum(d for _, d in product_cat_dims) + product_num_dim

        # MLP to combine all features into a single vector
        self.mlp = MLP(u_dim+p_dim, hidden_dims, dropout)

        last_size = _last_linear_out_features(self.mlp.net)
        self.head_detail_viewed = nn.Linear(last_size, 1)
        self.head_order_initiated = nn.Linear(last_size, 1)
        self.head_order_placed = nn.Linear(last_size, 1)
        # payment head predicts z, with exp(z) ≈ payment
        self.head_payment = nn.Linear(last_size, 1)

        self.alpha = alpha  # global downsampling ratio

    def forward(self, user_id, user_cat, user_num, product_id, product_cat, product_num, output_type='logit'):
        # Embed user IDs
        u_id = self.user_id_emb(user_id)
        # Embed and concatenate all categorical features (if any)
        if self.user_cat_embs:
            u_cat = torch.cat([emb(user_cat[:, i]) for i, emb in enumerate(self.user_cat_embs)], dim=-1)
        else:
            # If no categorical features, create empty tensor
            u_cat = torch.zeros(u_id.size(0), 0, device=u_id.device)

        # Embed product ID
        p_id = self.product_id_emb(product_id)
        # Embed and concatenate all categorical features (if any)
        if self.product_cat_embs:
            p_cat = torch.cat([emb(product_cat[:, i]) for i, emb in enumerate(self.product_cat_embs)], dim=-1)
        else:
            # If no categorical features, create empty tensor
            p_cat = torch.zeros(p_id.size(0), 0, device=p_id.device)

        # Concatenate all features and pass through MLP
        h = self.mlp(torch.cat([u_id, u_cat, user_num, p_id, p_cat, product_num], dim=-1))

        if output_type == 'logit':
            # Raw logit outputs
            detail_viewed_pred = self.head_detail_viewed(h).squeeze(-1)
            order_initiated_pred = self.head_order_initiated(h).squeeze(-1)
            order_placed_pred = self.head_order_placed(h).squeeze(-1)
            # payment time: transformation base on [P Covington, et.al. Deep Neural Networks for YouTube Recommendations. In RecSys, 2016]
            payment_pred = self.head_payment(h).squeeze(-1)
        elif output_type == 'prob':
            # Raw probablity outputs
            detail_viewed_pred = torch.sigmoid(self.head_detail_viewed(h).squeeze(-1))
            order_initiated_pred = torch.sigmoid(self.head_order_initiated(h).squeeze(-1))
            order_placed_pred = torch.sigmoid(self.head_order_placed(h).squeeze(-1))
            payment_pred = torch.sigmoid(self.head_payment(h).squeeze(-1))
        elif output_type == 'calibrated_prob':
            # Apply calibration to probablities when downsampling applied to imbalanced class
            # Xinran He et al. Practical lessons from predicting detail_vieweds on ads at Facebook. In the 8th International Workshop on Data Mining for Online Advertising.
            detail_viewed_pred = calibrate_probability(torch.sigmoid(self.head_detail_viewed(h).squeeze(-1)), self.alpha)
            order_initiated_pred = calibrate_probability(torch.sigmoid(self.head_order_initiated(h).squeeze(-1)), self.alpha)
            order_placed_pred = calibrate_probability(torch.sigmoid(self.head_order_placed(h).squeeze(-1)), self.alpha)
            payment_pred = torch.sigmoid(self.head_payment(h).squeeze(-1))  # no calibration for payment time
        else:
            ValueError("Output_type must be logit, prob or calibrated_prob")

        return {
            "detail_viewed": detail_viewed_pred,
            "order_initiated": order_initiated_pred,
            "order_placed": order_placed_pred,
            "payment": payment_pred,
        }

In [0]:
def model_inference(model, data_loader, weights, output_type='logit', device="cpu"):
    """Top-K recommendation function"""
    model.eval()
    
    user_ids = []
    product_ids = []
    all_scores = []

    with torch.no_grad():
        for batch in data_loader:
            inputs = (
                    batch['user_id'].to(device),
                    batch['user_cat'].to(device),
                    batch['user_num'].to(device),
                    batch['product_id'].to(device),
                    batch['product_cat'].to(device),
                    batch['product_num'].to(device)
            )
            preds = model(*inputs, output_type=output_type)
            if output_type == 'logit':
                # convert to probablity outputs
                preds_detail_viewed= torch.sigmoid(preds["detail_viewed"].squeeze())
                preds_order_initiated = torch.sigmoid(preds["order_initiated"].squeeze())
                preds_order_placed = torch.sigmoid(preds["order_placed"].squeeze())
            else:
                # assume probablity outputs
                preds_detail_viewed = preds["detail_viewed"].squeeze()
                preds_order_initiated = preds["order_initiated"].squeeze()
                preds_order_placed = preds["order_placed"].squeeze()
            
            # convert to estimated payment time
            preds_payment = torch.exp(preds["payment"].squeeze())
            # squashed estimated payment time to [0,1] using sigmoid
            preds_payment_norm = torch.sigmoid(preds_payment)

            scores = (
                weights.get('detail_viewed', 1)  * preds_detail_viewed.cpu().numpy()
                + weights.get('order_initiated', 1)  * preds_order_initiated.cpu().numpy()
                + weights.get('order_placed', 1) * preds_order_placed.cpu().numpy()
                + weights.get('payment', 1) * preds_payment_norm.cpu().numpy()
            )
            # Ensure it has shape (1,) if it's a scalar
            if scores.ndim == 0:
                scores = np.array([scores])
                
            user_ids.append(batch['user_id_int'].cpu())
            product_ids.append(batch['product_id_int'].cpu())
            # user_ids.append(batch['user_id_int'].cpu().numpy())
            # product_ids.append(batch['product_id_int'].cpu().numpy())
            all_scores.append(scores)

    # Concatenate
    user_ids = torch.cat(user_ids, dim=0)
    product_ids = torch.cat(product_ids, dim=0)
    # user_ids = np.concatenate(user_ids)
    # product_ids = np.concatenate(product_ids)
    y_scores = np.concatenate(all_scores)

    return {'user_int_id':user_ids, 'product_int_id':product_ids, 'score':y_scores} 

In [0]:
def run_streaming_inference(spark, db, table_names, run_date, 
                           data_args, model, weights, chunk_size, result_dir, 
                           batch_size=1024, shard_id=0, num_shards=1):
    """Complete streaming inference pipeline without disk I/O for chunks"""
    
    result_dir = resolve_local_result_dir(result_dir)
    Path(result_dir).mkdir(parents=True, exist_ok=True)

    # Load data
    user_df, product_df = load_user_product_spark_dataframes(
        spark, db, table_names, run_date
    )

    proc_res = process_user_product_feature_data(user_df, product_df, **data_args)
    user_df, product_df = proc_res['user_df'], proc_res['product_df']
    
    candidate_df = load_candidate_spark_dataframes(
        spark, db, table_names, run_date
    )

    candidate_df = build_shard_candidate_dataframe(candidate_df,
        run_date,
        shard_id=shard_id,
        num_shards=num_shards,
    )

    n_rows = candidate_df.count()
    n_users = candidate_df.select(F.countDistinct("user_id")).first()[0]
    n_products = candidate_df.select(F.countDistinct("product_id")).first()[0]
    
    print("\n" + "="*60)
    print("Data processing:")
    print("="*60)
    print(f"Users (inference):           {n_users}")
    print(f"products (inference):           {n_products}")
    print(f"Total rows (inference):      {n_rows}")  
    print("\n" + "="*60)

    # Save decoders
    pickle.dump(proc_res['uid_decoder'], open(Path(result_dir) / "uid_decoder.pkl", 'wb'))
    pickle.dump(proc_res['pid_decoder'], open(Path(result_dir) / "pid_decoder.pkl", 'wb'))
    del proc_res

    n_chunks = max(1, (n_rows + chunk_size - 1) // chunk_size)
    pipeline_started_at = perf_counter()
    
    for i, chunk_start_idx in enumerate(range(0, n_rows, chunk_size)):
        chunk_started_at = perf_counter()
        print(f"Run inference for chunk: {i + 1}/{n_chunks}")
        
        # 1. Load candidate_set chunk in Spark
        candidate_chunk_df = load_candidate_chunk(candidate_df, chunk_start_idx, chunk_size)
        
        # 2. Join with features in Spark
        joined_chunk_df = join_ranking_inference_data(user_df, product_df, candidate_chunk_df, data_args['user_id_col'], data_args['product_id_col'])
        
        # 3. Collect to Driver (Pandas)
        collect_started_at = perf_counter()
        pandas_chunk_df = joined_chunk_df.toPandas()
        collect_elapsed = perf_counter() - collect_started_at
        
        # 4. Create Loader from Memory
        data_loader = create_data_loader(data_source=pandas_chunk_df, batch_size=batch_size, **data_args)
        
        # 5. Run inference
        inference_started_at = perf_counter()
        chunk_res = model_inference(model, data_loader, weights, device="cpu")
        inference_elapsed = perf_counter() - inference_started_at
        
        # 6. Save results
        pickle.dump(chunk_res, open(Path(result_dir) / f"result_chunk_{i}.pkl", 'wb'))
        
        chunk_elapsed = perf_counter() - chunk_started_at
        avg_chunk_elapsed = (perf_counter() - pipeline_started_at) / max(1, i + 1)
        remaining_chunks = n_chunks - (i + 1)
        estimated_remaining_minutes = (avg_chunk_elapsed * remaining_chunks) / 60.0
        print(
            f"  rows={len(pandas_chunk_df)} collect_sec={collect_elapsed:.1f} "
            f"infer_sec={inference_elapsed:.1f} total_sec={chunk_elapsed:.1f} "
            f"eta_min={estimated_remaining_minutes:.1f}"
        )

        # Cleanup memory
        del candidate_chunk_df
        del joined_chunk_df
        del pandas_chunk_df
        del chunk_res
        del data_loader

    return {'n_users':n_users, 'n_products':n_products}

In [0]:
# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

table_config_path = dbutils.widgets.get("table_config_path")
with open(table_config_path, "r") as file:
    table_config = json.load(file)
db, table_names = table_config['DATABASE'], table_config['TABLE_NAMES']

model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)

infer_config_path = dbutils.widgets.get("infer_config_path")
with open(infer_config_path, "r") as file:
    infer_config = json.load(file)    

num_shards = int(dbutils.widgets.get("num_shards") or "1")
shard_id_raw = int(dbutils.widgets.get("shard_id") or "0")
shard_id_base = dbutils.widgets.get("shard_id_base") or "0"

shard_id = normalize_shard_id(
    shard_id_raw,
    num_shards,
    shard_id_base,
)
print(f"Input parameter num_shards:{num_shards}")
print(f"Input parameter shard_id_raw:{shard_id_raw}")
print(f"Input parameter shard_id_base:{shard_id_base}")
print(f"Normalized shard_id:{shard_id}")

In [0]:
# config parameters
data_args = dict(
        user_id_col='user_id', 
        product_id_col='product_id',
        user_cat_cols=["platform"],
        user_num_cols=['store_visits_30d',
        'store_visits_90d',
        'pdp_views_30d',
        'pdp_views_90d',
        'tot_items_ordered_30d',
        'tot_items_ordered_90d',
        'tot_items_ordered_180d',
        'app_items_ordered_30d',
        'app_items_ordered_90d',
        'app_items_ordered_180d',
        'coupons_owned_30d',
        'coupons_owned_90d',
        'coupons_owned_180d',
        'coupons_redeemed_30d',
        'coupons_redeemed_90d',
        'coupons_redeemed_180d'],
        product_cat_cols=["product_type"], 
        product_num_cols=['card_uv_imps_30d',
        'card_uv_imps_90d',
        'pdp_uv_imps_30d',
        'pdp_uv_imps_90d',
        'tot_order_cnt_30d',
        'tot_order_cnt_90d',
        'app_order_cnt_30d',
        'app_order_cnt_90d',
        'avg_order_msrp',
        'avg_order_baseprice_60d',
        'avg_trans_price_30d',
        'avg_trans_price_90d']
)

model_args = model_config['model_args']

metadata_dir = model_config['METADATA_PATH']

neg_ratio = model_config['neg_sample_ratio']
batch_size = model_config['batch_size']
weights = model_config['task_weights']

input_chunk_size = infer_config['input_data_chunk_size']

model_dir = model_config["MODEL_PATH"]
result_root_dir = resolve_local_result_dir(model_config['RESULT_PATH'])
result_dir = result_root_dir
if num_shards > 1:
    result_dir = str(Path(result_dir) / f"shard_{shard_id}")

In [0]:
metadata = pickle.load(open(Path(metadata_dir) / 'metadata_stats.pkl', 'rb'))
user_cat_dims = [(d, min(model_args['cat_emb_dim'], d)) for d in metadata['user_cat_dims']]
product_cat_dims = [(d, min(model_args['cat_emb_dim'], d)) for d in metadata['product_cat_dims']]
model = MultiTaskRanker(uid_vocab=metadata["n_users"], 
                        uid_dim=model_args['id_emb_dim'],
                        user_cat_dims=user_cat_dims,
                        user_num_dim=metadata['user_num_dim'],
                        pid_vocab=metadata["n_products"], 
                        pid_dim=model_args['id_emb_dim'],
                        product_cat_dims=product_cat_dims,
                        product_num_dim=metadata['product_num_dim'],
                        hidden_dims=model_args['hidden_dims'], 
                        dropout=model_args['dropout'],
                        alpha=neg_ratio)

model.load_state_dict(torch.load(Path(model_dir).joinpath("model"), weights_only=True))
model.eval()                        

In [0]:
# clean all existing ranking result artifacts from the volume before running the pipeline
# In parallel mode, clean only the current shard folder to avoid wiping sibling shard artifacts.
# In single-shard mode, clean the full root as before.
if num_shards > 1:
    clear_result_dir(result_dir)
else:
    clear_result_dir(result_root_dir)

In [0]:
infer_stats = run_streaming_inference(spark, 
                                    db, 
                                    table_names, 
                                    run_date, 
                                    data_args, 
                                    model, 
                                    weights, 
                                    input_chunk_size, 
                                    result_dir, 
                                    batch_size,
                                    shard_id=shard_id,
                                    num_shards=num_shards)

In [0]:
# ============================================================
# Write result
# ============================================================
def transform_dict(result_dict, K):
    # ---- 1. Flatten all tensors/arrays ----
    user_all = torch.cat(result_dict["user_int_id"]).numpy()
    product_all = torch.cat(result_dict["product_int_id"]).numpy()
    score_all = np.concatenate(result_dict["score"])

    # ---- 2. Create distinct lists ----
    unique_users, user_inverse = np.unique(user_all, return_inverse=True)
    unique_products, product_inverse = np.unique(product_all, return_inverse=True)

    # ---- 3. Group score+product by user ----
    user_to_scores = defaultdict(list)
    user_to_productidx = defaultdict(list)

    for i, u_idx in enumerate(user_inverse):
        user_to_scores[u_idx].append(score_all[i])
        user_to_productidx[u_idx].append(product_inverse[i])

    # ---- 4. Compute top-K scores + indices per user ----
    topk_scores = []
    topk_indices = []

    for u_idx in range(len(unique_users)):
        scores = np.array(user_to_scores[u_idx])
        product_indices = np.array(user_to_productidx[u_idx])

        if len(scores) == 0:
            topk_scores.append(np.array([]))
            topk_indices.append(np.array([]))
            continue

        # Sort descending
        sort_idx = np.argsort(-scores)

        topk = sort_idx[:K]
        topk_scores.append(scores[topk])
        topk_indices.append(product_indices[topk])

    # ---- 5. Construct new dict ----
    new_dict = {
        "user_int_ids": unique_users,
        "product_int_ids": unique_products,
        "topk_scores": np.array(topk_scores),
        "topk_indices": np.array(topk_indices)
    }

    return new_dict

def _decode_ids(encoded_ids, decoder, to_numpy=False):
    ids = [decoder[id] for id in encoded_ids]
    return np.array(ids) if to_numpy else ids

def retrieve_k_products_per_user(
    spark,
    model_res,
    decoder_uid,
    decoder_pid,
    top_k,
    output_table_name,
    run_date,
    chunk_size=5000,
    shard_id=0,
    num_shards=1
):
    uids = model_res['user_int_ids']
    pids = model_res['product_int_ids']
    top_ind = model_res['topk_indices']
    top_scores = model_res['topk_scores']
    top_ind = top_ind[:,:top_k]
    top_scores = top_scores[:,:top_k]

    uids = _decode_ids(uids, decoder_uid)
    pids = _decode_ids(pids, decoder_pid, to_numpy=True)

    # define schema
    schema = StructType([
        StructField("user_id", StringType(), False),
        StructField("product_id", ArrayType(StringType()), False),
        StructField("score", ArrayType(DoubleType()), False)
    ])

    # Process data in chunks to avoid memory issues
    wrote_any_rows = False
    for i in range(0, len(uids), chunk_size):
        chunk_data = []
        end_idx = min(i + chunk_size, len(uids))

        for j in range(i, end_idx):
            user_id = uids[j]
            indices = top_ind[j]
            products = pids[np.asarray(indices, dtype=np.int64)[:top_k]].tolist()
            scores = top_scores[j][:top_k].tolist()
            chunk_data.append((user_id, products, scores))

        chunk_df = spark.createDataFrame(chunk_data, schema=schema)

        invalid_product_count = chunk_df.filter(F.size("product_id") != top_k).count()
        if invalid_product_count > 0:
            raise RuntimeError(
                f"Failed to build exactly {top_k} products for {invalid_product_count} users in ranking output chunk starting at index {i}"
            )

        chunk_df = chunk_df.withColumn("partition_date", F.lit(run_date))

        if not wrote_any_rows:
            chunk_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(output_table_name)
            wrote_any_rows = True
        else:
            chunk_df.write.mode("append").saveAsTable(output_table_name)

    return None

In [0]:
# write candidate_set results to table
model_res = load_and_merge_results(result_dir)
top_k = min(infer_stats["n_products"], infer_config["top_k_products"])
table_write_chunk_size = infer_config["table_write_chunk_size"]
results = transform_dict(model_res, top_k)

In [0]:
uid_decoder, pid_decoder = load_decoders(result_dir)
ranking_list_table_name = (
    db + '.' + table_names['mmc_ranking_list_results'] + f"_parallel_temp_{shard_id}"
    if num_shards > 1
    else db + '.' + table_names['mmc_ranking_list_results']
)
print("Write ranking lists to table:")
retrieve_k_products_per_user(spark,
                            results,
                            uid_decoder,
                            pid_decoder,
                            top_k=top_k,
                            output_table_name=ranking_list_table_name,
                            run_date=run_date,
                            chunk_size=table_write_chunk_size,
                            shard_id=shard_id,
                            num_shards=num_shards)                            

In [0]:
# convert collected lists to flattened format and map user_id to vin
def transform_result_table(spark, db, table_names, run_date):

    ranking_list_table = (
        db + '.' + table_names['mmc_ranking_list_results'] + f"_parallel_temp_{shard_id}"
        if num_shards > 1
        else db + '.' + table_names['mmc_ranking_list_results']
    )  
    result_table = (
        db + '.' + table_names['mmc_final_results'] + f"_parallel_temp_{shard_id}"
        if num_shards > 1
        else db + '.' + table_names['mmc_final_results']
    )
    # dedupe the results as user_ids and VINs have many-to-many mapping
    result_df = spark.sql("""
                          select vum.vin, rl.product_id, p.product_type, max(rl.score) as score
                          from (
                            select
                              user_id,
                              item.product_id AS product_id,
                              item.score AS score
                            from {ranking_list_table}
                            LATERAL VIEW explode(arrays_zip(product_id, score)) t AS item
                            where partition_date = '{run_date}'
                          ) rl
                          inner join (
                            select distinct vin, ciamid as user_id
                            from {db}.{vin_meid_mapping} 
                            where partition_date <= '{run_date}'
                          ) vum
                          on vum.user_id = rl.user_id
                          inner join {db}.{mmc_product} p
                          on p.product_id = rl.product_id
                          group by 1, 2, 3
                          """.format(db = db,
                            vin_meid_mapping=table_names['vin_meid_mapping'],
                            mmc_product=table_names['mmc_product'],
                            ranking_list_table = ranking_list_table,
                            run_date = run_date))

    result_df = result_df.withColumn("dt", F.lit(run_date))

    result_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(result_table)

    return None



In [0]:
transform_result_table(spark, db, table_names, run_date)